# 01. Baseline SARIMAX

제조업체별·중분류별 SARIMAX 기본 모델 실험 노트북입니다.

- 제조업체 단위 예측의 한계를 확인
- 중분류(라면류, 면류 등) 단위 대안 모델 탐색
- 이후 클러스터링 접근으로 전환하는 계기가 된 초기 실험

In [20]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [21]:
final = pd.read_csv('../data/final.csv', encoding='utf-8-sig')
final.head()

,판매월,제조업체,상품명,판매수량,대분류,중분류,한파,폭염,호우,코로나확진자수,생활물가지수,음식료품_계절조정지수
0,2021-01,88010455,(오뚜기)열라면종이(용기)105g,1,면류.라면류,라면류,22,0,0,17470.0,101.2,105.5
1,2021-01,88030600,(백제물산)즉석쌀국수멸치맛92g,2,면류.라면류,면류,22,0,0,17470.0,101.2,105.5
2,2021-01,88030600,(백제물산)즉석쌀국수멸치맛92g,1,면류.라면류,면류,22,0,0,17470.0,101.2,105.5
3,2021-01,88010455,(오뚜기)열라면종이(용기)105g,2,면류.라면류,라면류,22,0,0,17470.0,101.2,105.5
4,2021-01,88010455,(오뚜기)열라면종이(용기)105g,1,면류.라면류,라면류,22,0,0,17470.0,101.2,105.5


---

### 독립변수 조정
- 제조업체별 예측 ( ~제조업체별 판매횟수~ or 판매수량)
- 중분류별 예측 (라면류, 면류, 기타면류)
- 코로나 데이터 유무 (default : 제외하고, 위에서 성능 좋았던 경우에 추가)
- ~휴가 관련 데이터 추가~

---
### 모델 조정
- SARIMA 기반 예측
- 트리기반 모델 앙상블 (시계열 + 회귀 가능한 모델 search)

---
### 파라미터 조정
- 

---

## 01. 제조업체별 예측

In [22]:
final

,판매월,제조업체,상품명,판매수량,대분류,중분류,한파,폭염,호우,코로나확진자수,생활물가지수,음식료품_계절조정지수
0,2021-01,88010455,(오뚜기)열라면종이(용기)105g,1,면류.라면류,라면류,22,0,0,17470.0,101.20,105.5
1,2021-01,88030600,(백제물산)즉석쌀국수멸치맛92g,2,면류.라면류,면류,22,0,0,17470.0,101.20,105.5
2,2021-01,88030600,(백제물산)즉석쌀국수멸치맛92g,1,면류.라면류,면류,22,0,0,17470.0,101.20,105.5
3,2021-01,88010455,(오뚜기)열라면종이(용기)105g,2,면류.라면류,라면류,22,0,0,17470.0,101.20,105.5
4,2021-01,88010455,(오뚜기)열라면종이(용기)105g,1,면류.라면류,라면류,22,0,0,17470.0,101.20,105.5
...,...,...,...,...,...,...,...,...,...,...,...,...
80572,2023-12,88010430,(농심)안성탕면컵6입,1,면류.라면류,라면류,14,0,2,274213.6,114.84,94.6
80573,2023-12,88010455,(오뚜기)진라면순한맛컵6입,3,면류.라면류,라면류,14,0,2,274213.6,114.84,94.6
80574,2023-12,88030600,(백제물산)즉석쌀국수얼큰맛92g,10,면류.라면류,면류,14,0,2,274213.6,114.84,94.6
80575,2023-12,88010430,(농심)육개장사발면컵6입,1,면류.라면류,라면류,14,0,2,274213.6,114.84,94.6


In [23]:
final_1 = final.copy()
final_1.drop(columns=['상품명', '대분류', '중분류'], inplace=True)

In [24]:
final_1.head()

,판매월,제조업체,판매수량,한파,폭염,호우,코로나확진자수,생활물가지수,음식료품_계절조정지수
0,2021-01,88010455,1,22,0,0,17470.0,101.2,105.5
1,2021-01,88030600,2,22,0,0,17470.0,101.2,105.5
2,2021-01,88030600,1,22,0,0,17470.0,101.2,105.5
3,2021-01,88010455,2,22,0,0,17470.0,101.2,105.5
4,2021-01,88010455,1,22,0,0,17470.0,101.2,105.5


In [25]:
final_1['판매월'] = pd.to_datetime(final_1['판매월']).dt.to_period('M')
filtered = final_1[['판매월', '한파', '폭염', '호우', '코로나확진자수', '생활물가지수', '음식료품_계절조정지수']]
other = filtered.groupby('판매월').mean().reset_index()
other

,판매월,한파,폭염,호우,코로나확진자수,생활물가지수,음식료품_계절조정지수
0,2021-01,22.0,0.0,0.0,17470.0,101.20,105.5
1,2021-02,12.0,0.0,0.0,11466.0,102.11,97.1
2,2021-03,0.0,0.0,9.0,13414.0,102.53,98.7
3,2021-04,2.0,0.0,0.0,18926.0,102.65,101.8
4,2021-05,0.0,0.0,11.0,18329.0,102.67,99.6
5,2021-06,0.0,0.0,9.0,16623.0,102.80,100.6
6,2021-07,0.0,77.0,130.0,41372.0,102.77,102.6
7,2021-08,0.0,34.0,179.0,53073.0,103.33,101.8
8,2021-09,0.0,0.0,32.0,59850.0,104.23,105.6
9,2021-10,5.0,0.0,4.0,53407.0,104.42,102.8


In [26]:
sales = final_1.groupby(['판매월', '제조업체'], as_index=False)['판매수량'].sum()

In [27]:
final_1 = pd.merge(sales, other, on='판매월', how='outer')
final_1

,판매월,제조업체,판매수량,한파,폭염,호우,코로나확진자수,생활물가지수,음식료품_계절조정지수
0,2021-01,88010249,4,22.0,0.0,0.0,17470.0,101.20,105.5
1,2021-01,88010430,3719,22.0,0.0,0.0,17470.0,101.20,105.5
2,2021-01,88010438,127,22.0,0.0,0.0,17470.0,101.20,105.5
3,2021-01,88010453,450,22.0,0.0,0.0,17470.0,101.20,105.5
4,2021-01,88010454,229,22.0,0.0,0.0,17470.0,101.20,105.5
...,...,...,...,...,...,...,...,...,...
629,2023-12,88012772,3,14.0,0.0,2.0,274213.6,114.84,94.6
630,2023-12,88023490,3,14.0,0.0,2.0,274213.6,114.84,94.6
631,2023-12,88030600,559,14.0,0.0,2.0,274213.6,114.84,94.6
632,2023-12,88092968,3,14.0,0.0,2.0,274213.6,114.84,94.6


In [28]:
final_1[final_1['판매월']=='2021-01']

,판매월,제조업체,판매수량,한파,폭염,호우,코로나확진자수,생활물가지수,음식료품_계절조정지수
0,2021-01,88010249,4,22.0,0.0,0.0,17470.0,101.2,105.5
1,2021-01,88010430,3719,22.0,0.0,0.0,17470.0,101.2,105.5
2,2021-01,88010438,127,22.0,0.0,0.0,17470.0,101.2,105.5
3,2021-01,88010453,450,22.0,0.0,0.0,17470.0,101.2,105.5
4,2021-01,88010454,229,22.0,0.0,0.0,17470.0,101.2,105.5
5,2021-01,88010455,2673,22.0,0.0,0.0,17470.0,101.2,105.5
6,2021-01,88010456,53,22.0,0.0,0.0,17470.0,101.2,105.5
7,2021-01,88010520,12,22.0,0.0,0.0,17470.0,101.2,105.5
8,2021-01,88010731,275,22.0,0.0,0.0,17470.0,101.2,105.5
9,2021-01,88010732,106,22.0,0.0,0.0,17470.0,101.2,105.5


In [29]:
final_1['제조업체'].nunique()

29

In [30]:
final_1['제조업체'].value_counts().tail(10) #제일 개수가 적은 하위 10개만 출력

88023490    14
88010072    13
88010733    12
88012772     9
88010070     6
88010450     5
88012773     4
69728925     4
88012998     2
88010451     2
Name: 제조업체, dtype: int64

In [48]:
total = final_1['판매수량'].sum()

In [49]:
316 / total

0.0012402565290086582

In [51]:
final_1[final_1['제조업체']==88010527]['판매수량'].sum()

168

In [56]:
final_1.groupby('제조업체')['판매수량'].sum().reset_index().sort_values(by='판매수량', ascending=False)

,제조업체,판매수량
5,88010430,106999
11,88010455,69525
25,88030600,26296
10,88010454,13272
16,88010731,9620
19,88011285,7499
9,88010453,6270
17,88010732,4617
6,88010438,3838
28,88096952,1796


In [15]:
final_1['제조업체'].value_counts()

88010438    36
88010453    36
88010454    36
88010455    36
88010731    36
88010732    36
88011285    36
88030600    36
88010430    36
88010456    35
88092968    34
88010249    32
88012770    25
88096952    24
88010527    22
88010457    21
88093891    16
88010520    16
88010399    14
88023490    14
88010072    13
88010733    12
88012772     9
88010070     6
88010450     5
88012773     4
69728925     4
88012998     2
88010451     2
Name: 제조업체, dtype: int64

---

In [16]:
# 제조사별 데이터프레임 생성 및 예측 함수 정의

def sarima_forecast(df, manufacturer):
    # 해당 제조업체 데이터만 선택
    manufacturer_data = df[df['제조업체'] == manufacturer].set_index('판매월').resample('M').sum()

    # 데이터가 존재하지 않는 경우 처리
    if manufacturer_data.empty:
        print(f"제조사 '{manufacturer}'의 데이터가 없습니다. 예측을 건너뜁니다.")
        return None, None

    # 훈련 데이터와 테스트 데이터 분리
    train = manufacturer_data.loc[:'2023-08']
    test = manufacturer_data.loc['2023-09':]

    # 테스트 데이터가 비어 있는 경우 처리
    if test.empty or len(train) < 5:  # 훈련 데이터가 5개 미만일 경우 예측 건너뛰기
        print(f"제조사 '{manufacturer}'의 테스트 데이터가 없습니다. 예측을 건너뜁니다.")
        return None, None

    # SARIMA 모델 정의 (p, d, q) x (P, D, Q, s)
    model = SARIMAX(train['판매수량'], 
                    exog=train[['한파', '폭염', '호우', '코로나확진자수', '생활물가지수', '음식료품_계절조정지수']], 
                    order=(1, 1, 1), 
                    seasonal_order=(1, 1, 1, 12))

    # 모델 학습
    model_fit = model.fit(disp=False)

    # 예측을 위한 시작 시점 설정 (훈련 데이터의 마지막 시점 + 1)
    start = test.index[0]  # 테스트 데이터의 첫 번째 인덱스를 예측 시작 시점으로 사용
    end = test.index[-1]  # 테스트 데이터의 마지막 인덱스를 예측 종료 시점으로 사용

    # 예측
    forecast = model_fit.get_forecast(steps=len(test), exog=test[['한파', '폭염', '호우', '코로나확진자수', '생활물가지수', '음식료품_계절조정지수']])
    forecast_values = forecast.predicted_mean
    return forecast_values, test['판매수량']

In [17]:
# 제조사와 그 개수 세기
manufacturer_counts = final_1['제조업체'].value_counts()

# manufacturers 리스트 초기화
manufacturers = []

# 개수가 한 자리 수인 경우 99999999에 포함
for manufacturer, count in manufacturer_counts.items():
    if count < 10:  # 한 자리 수인 경우
        manufacturers.append(99999999)  # 기타 제조사 추가
    else:
        manufacturers.append(manufacturer)  # 제조사 추가

# 중복 제거
manufacturers = list(set(manufacturers))

# 모든 제조사에 대한 예측 결과 저장
predictions = {}
final_results = pd.DataFrame()

In [18]:
# 제조사와 그 개수 세기
manufacturer_counts = final_1['제조업체'].value_counts()

# manufacturers 리스트 초기화
manufacturers = []

# 개수가 한 자리 수인 경우 99999999에 포함
for manufacturer, count in manufacturer_counts.items():
    if count < 10:  # 한 자리 수인 경우
        manufacturers.append(99999999)  # 기타 제조사 추가
    else:
        manufacturers.append(manufacturer)  # 제조사 추가

# 중복 제거
manufacturers = list(set(manufacturers))

# 모든 제조사에 대한 예측 결과 저장
predictions = {}
final_results = pd.DataFrame()

for manufacturer in manufacturers:
    forecast_values, actual_values = sarima_forecast(final_1, manufacturer)
    
    # 예측 결과가 None이 아닌 경우에만 추가
    if forecast_values is not None and actual_values is not None:
        predictions[manufacturer] = forecast_values
        
        # 제조사별 예측 결과 DataFrame 생성
        manufacturer_result = pd.DataFrame({
            '판매월': actual_values.index,
            '제조업체': manufacturer,
            '예측판매수량': forecast_values,
            '실제판매수량': actual_values
        })

        # 최종 결과에 추가
        final_results = pd.concat([final_results, manufacturer_result])

# 최종 월별 예측 결과
final_monthly_results = final_results.groupby('판매월').agg({
    '예측판매수량': 'sum',
    '실제판매수량': 'sum'
}).reset_index()

# 평가지표 계산
if not final_monthly_results.empty:
    mse = mean_squared_error(final_monthly_results['실제판매수량'], final_monthly_results['예측판매수량'])
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(final_monthly_results['실제판매수량'], final_monthly_results['예측판매수량'])

    print(f'Mean Squared Error: {mse:.2f}')
    print(f'Root Mean Squared Error: {rmse:.2f}')
    print(f'Mean Absolute Error: {mae:.2f}')

# 결과 확인
final_monthly_results

C:\ProgramData\Anaconda3\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
C:\ProgramData\Anaconda3\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


MissingDataError: exog contains inf or nans

---

## 02. 중분류별 예측

In [ ]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

In [64]:
final_2 = final.copy()

In [65]:
final_2 = final.copy()

# 판매월을 datetime 형식으로 변환
final_2['판매월'] = pd.to_datetime(final_2['판매월'], format='%Y-%m')

# 중분류별로 월별 판매량 합계 구하기
final_2 = final_2.groupby(['판매월', '중분류'])['판매수량'].sum().unstack()

In [66]:
final_2.head()

중분류,기타라면.면류,라면류,면류
판매월,,,
2021-01-01,57,7015,1736
2021-02-01,18,4338,1387
2021-03-01,12,7779,1437
2021-04-01,10,5486,1445
2021-05-01,7,5674,1289


In [69]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
import numpy as np

# SARIMA 모델을 사용하여 예측하는 함수
def sarima_forecast(train, steps, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)):
    model = SARIMAX(train, order=order, seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False)
    model_fit = model.fit(disp=False)
    
    # 예측을 위해 steps(2023년 9월부터 12월까지 4개월 예측)를 사용
    forecast = model_fit.forecast(steps=steps)
    
    return forecast

# 예측할 중분류 선택 
target_category = ['라면류', '면류', '기타라면.면류']

# 중분류별로 예측 수행
predictions = {}

for category in target_category:
    # 해당 중분류 데이터만 선택
    category_data = final_2[category].dropna()
    
    # 훈련 데이터로 모든 데이터를 사용 (2023년 8월까지)
    train = category_data
    
    # SARIMA 모델을 사용하여 4개월(2023년 9월 ~ 12월) 예측 수행
    forecast_values = sarima_forecast(train, steps=4)
    
    # 예측 결과 저장
    predictions[category] = forecast_values
    
    # 예측 결과 출력 (2023년 9월부터 12월까지)
    future_months = pd.date_range(start='2023-09-01', periods=4, freq='M')
    result = pd.DataFrame({
        '판매월': future_months,
        '예측판매수량': forecast_values
    })
    
    print(f"\n중분류: {category}")
    print(result)



중분류: 라면류
                  판매월       예측판매수량
2024-01-01 2023-09-30  5759.825500
2024-02-01 2023-10-31  4424.755494
2024-03-01 2023-11-30  6094.492053
2024-04-01 2023-12-31  4861.825997

중분류: 면류
                  판매월       예측판매수량
2024-01-01 2023-09-30  1332.019515
2024-02-01 2023-10-31  1664.311994
2024-03-01 2023-11-30  1188.420320
2024-04-01 2023-12-31  1479.259214

중분류: 기타라면.면류
                  판매월      예측판매수량
2024-01-01 2023-09-30   55.827168
2024-02-01 2023-10-31   71.232533
2024-03-01 2023-11-30  141.047886
2024-04-01 2023-12-31  120.519589
